---
title: "07. LLM release artifacts"
description: "Package an LLM app as an MLflow pyfunc version, evaluate it with a repeatable one-shot job on the local Compose stack, and release it through the same promotion path as any other model."
---

## Outcome

An LLM workflow (prompt + external model endpoint + retrieval/config) is packaged as a
single MLflow `pyfunc` model version, evaluated by a repeatable evaluator job, and released
through the **same promotion mechanism as any other model**. There is no separate LLM
control path: an LLM app is just another kind of version in the registry built in
[chapter 03](03-reproducible-training.ipynb), promoted exactly as [chapter 05](05-online-serving.ipynb)
promotes a classical model, and served and batch-scored by machinery chapters 04–06 already
built. Every step in this chapter runs against the local Docker Compose stack from
[chapter 02](02-local-foundation.ipynb).


## Design — an LLM app is a pyfunc version

The `pyfunc` artifact bundles everything that defines the app:

- the prompt template(s) and configuration,
- the model/endpoint reference and generation parameters,
- any retrieval/index configuration,
- the signature (inputs/outputs) and dependencies.

Registering it produces a **version number** in the same registry classical models
use. Serving and batch inference load it by `models:/<name>/<version>` exactly like
any other model.

- **Evaluation is a repeatable job.** A one-shot job scores the candidate version
  against a fixed evaluation set with defined metrics (exact match where expected
  outputs exist, plus latency and token cost); results are logged to MLflow and
  summarized in a results-DB record ([chapter 04](04-results-db-and-batch.ipynb)),
  and promotion requires meeting recorded thresholds. Locally it is the same
  one-shot pattern the demo stack runs train/batch with; Part II turns the same
  image into an ACA Job ([chapter 11](11-porting-to-aca.ipynb)).
- **Config travels inside the artifact; secrets do not.** Prompts/config live in
  the artifact so a version is self-contained. The external API key resolves at
  predict time from the environment (`MODEL_API_KEY`) when running locally; Part II
  replaces that variable with a Key Vault read via managed identity
  ([chapter 10](10-azure-foundation.ipynb)). Under either backend the key is never
  embedded in the artifact or image.
- **No bespoke release ledger.** A release is: registered version + its evaluation
  record + a Git tag + the image digest that references it. Same provenance chain,
  same registry, same promotion semantics as every other model
  ([chapter 08](08-environment-contract.ipynb)).


## Build in `projects/ml-platform/`

```
projects/ml-platform/
├── src/ml_platform/llm/
│   ├── model.py                # LLMPyfunc(PythonModel): load prompt/config artifacts,
│   │                           #   call OpenAI-compat endpoint, return uniform response
│   ├── artifact_builder.py     # build_and_register(): log prompt.yaml + config.yaml,
│   │                           #   package + register the pyfunc version
│   └── evaluator.py            # score candidate vs fixed eval JSONL → exact_match,
│   │                           #   latency, token counts; gates promotion
└── src/train_job/
    └── register_llm.py         # CLI entrypoint: configure_mlflow → build_and_register
                                #   → record_run (same pattern as train.py)
```

No new service joins the Compose stack: an LLM version is loaded by
`models:/name/version` and served / batch-scored identically to a classical model.
The LLM-specific code lives entirely in `ml_platform/llm/`; the rest of the platform
is unchanged. Both entrypoints (`register_llm.py`, `evaluator.py`) obey the same
one-shot job contract as train/batch: read configuration from the environment, talk
to `MLFLOW_TRACKING_URI`, write their results-DB record, exit non-zero on failure.

The external API key resolves at predict time: `MODEL_API_KEY` from the environment
first (export it before triggering the job locally), then a Key Vault fallback gated
on `KEY_VAULT_URL` that stays dormant until Part II wires it
([chapter 11](11-porting-to-aca.ipynb)). With neither value present the call fails
fast rather than issuing an unauthenticated request. The key is **never embedded in
the artifact or image**.


## How the pieces connect

### Pyfunc artifact (`ml_platform/llm/model.py`)

`LLMPyfunc` is an `mlflow.pyfunc.PythonModel` subclass with two methods:

- **`load_context`:** reads `prompt.yaml` and `config.yaml` from the artifact
  directory, populates `_system_prompt`, `_user_template`, `_endpoint`, and
  `_gen_params`. No network calls at load time; credentials are resolved lazily
  at predict time.
- **`predict`:** formats each `input` row via the user template, calls
  `_call_openai_compat` (a thin `httpx.post` against the `/chat/completions`
  endpoint), returns a DataFrame with `content`, `model`, `prompt_tokens`,
  `completion_tokens`.

The credential lookup in `_resolve_api_key` is strict: `MODEL_API_KEY` env wins;
otherwise the code fetches the Key Vault secret named by `MODEL_API_KEY_SECRET`
(default `model-api-key`) with `DefaultAzureCredential`, but only when
`KEY_VAULT_URL` is set. With neither variable present `predict` raises immediately;
there is no local mock and no anonymous call. Locally the whole setup is exporting
`MODEL_API_KEY` before the job runs. The Key Vault branch stays dormant until Part II
wires it: the foundation provisions the vault and grants the workload identities read
access ([chapter 10](10-azure-foundation.ipynb)), and the Job definition supplies
`KEY_VAULT_URL` ([chapter 11](11-porting-to-aca.ipynb)). The artifact contains no
secret values either way.

### Artifact builder (`ml_platform/llm/artifact_builder.py`)

`build_and_register(registered_name, prompt_yaml_path=…, config_yaml_path=…)`
opens an MLflow run, logs the prompt/config files as lineage artefacts, records
`model_endpoint`, `model_id`, and `temperature` as params (no secrets), and calls
`mlflow.pyfunc.log_model` with the `LLMPyfunc` instance, both artifact paths, and
the enforced input/output signature. Returns the `ModelVersion`. Omitting both paths
writes built-in stub prompt/config files, so producing a throwaway version needs no
external account.

A stub `canary_predict(model_uri)` loads the registered version and runs one
prediction, the same pattern as `evaluate.py`'s held-out check.

### Evaluator (`ml_platform/llm/evaluator.py`)

A one-shot job: run against the local Compose stack like train/batch (the runner
service that triggers those jobs is its natural trigger point,
[chapter 02](02-local-foundation.ipynb)), and an ACA Job in Part II
([chapter 11](11-porting-to-aca.ipynb)). It:

1. Loads `models:/<name>/<version>` via `mlflow.pyfunc.load_model`.
2. Reads a JSONL eval file (`{"input": "…", "expected": "…"}`); rows may omit `expected`.
3. Runs all predictions, measures wall-clock latency and token counts.
4. Computes `exact_match` over rows where `expected` is provided.
5. Logs all metrics plus a per-row CSV to the MLflow run; sets
   `gate_result=PASS|FAIL` as a run tag.
6. Applies threshold gates (`--min-exact-match`, `--max-avg-tokens`); exits
   non-zero on a miss so promotion is blocked.
7. Writes its record via `record_run`, so the dashboard lists this job the same way
   it lists a training run ([chapter 04](04-results-db-and-batch.ipynb),
   [chapter 06](06-observability-and-dashboard.ipynb)).

### Registration entrypoint (`src/train_job/register_llm.py`)

Thin CLI mirroring `train.py`: `configure_mlflow` → `build_and_register` →
results-DB record, with the run tagged from `IMAGE_DIGEST` exactly as training tags
its runs ([chapter 03](03-reproducible-training.ipynb)). The shared `ml_platform`
package (including `llm/`) is already copied into the train-job image wholesale, so
shipping this entrypoint alongside `train.py`/`evaluate.py` is one more `COPY` line,
or it rides in a sibling image if its dependencies diverge.

### No new services

Serving (`serving_app/app.py`) and batch scoring (`batch_job/score.py`) call
`mlflow.sklearn.load_model` today but widen to `mlflow.pyfunc.load_model`: a one-line
change, same `models:/name/version` URI (the batch worker already loads pyfunc). The
registry, promotion path, and results DB are identical for classical and LLM versions.


## Golden-path position & acceptance evidence

This chapter feeds a second kind of producer into the *same*
`register → eval → promote → serve/batch` path; no new branch appears. Promotion of
an LLM version is deliberately not restated here: it is the alias flip plus pinned
redeploy defined in [chapter 05](05-online-serving.ipynb) and tabulated for both
backends in [chapter 08](08-environment-contract.ipynb), wrapped by
`python demo/promote.py --model-name <llm-app> --version N` (local backend; the same
script's `--backend aca` is the Part II form).

**Acceptance evidence:**

- An LLM app registers as a pyfunc version and loads via `models:/name/version` in
  both the serving container and a batch job with no serving/batch code change.
- The evaluator runs as a one-shot job against the local stack, logs metrics to
  MLflow plus a results-DB record, and its thresholds gate promotion.
- Credentials enter through the environment at trigger time; the artifact and image
  contain no secrets. Part II changes only how that variable is delivered: managed
  identity plus Key Vault ([chapter 10](10-azure-foundation.ipynb),
  [chapter 11](11-porting-to-aca.ipynb)), same images otherwise.


## Extensions (deferred from the MVP)

| Deferred | Contract | MVP substitute |
|---|---|---|
| Rich evaluator evidence contract | `docs/03` | Threshold pass/fail + logged metrics |
| Provider/model upgrade workflow | `docs/03` | Re-register a new pyfunc version |
| Retrieval index lifecycle | `docs/03` | Static index config in the artifact |

## Why not Azure Machine Learning's built-in MLflow?

The demo pins `mlflow==3.15.1`, the latest open-source release at the time of
writing. Moving up from 2.x buys LoggedModel lineage, GenAI evaluation judges,
tracing, and prompt tooling; nothing taught here depends on those. Part II
could instead track against an Azure Machine Learning workspace, whose endpoint
emulates the MLflow REST API without hosting a server. As of writing its
plugin caps clients at `mlflow-skinny<=3.13.0`, and the supported surface is
narrower than what we run:

| Capability | Self-hosted OSS 3.15.1 (this platform) | MLflow APIs on Azure Machine Learning |
|---|---|---|
| Tracking runs, metrics, params, artifacts | Full | Core surface works |
| Run search and filter grammar | Full, including `OR` and order-by-metrics | Subset: no `OR`, no ordering by metrics, params, or tags |
| Registry versions and tags | Full | Yes, but models are immutable: no rename, no container delete |
| Aliases (`production` flip) | Core of our promotion contract | Absent from Azure ML's documented surface; its docs teach legacy stages instead |
| Legacy stages | Deprecated in 3.x | Documented mechanism, SDK-only, invisible to Studio and CLI |
| LoggedModel lineage entity | Yes | No |
| GenAI evaluation, Prompt Registry, AI Gateway | Yes | No |
| Tracing UI and cost tracking | Yes | No |

The alias row decides it. Promotion everywhere in this course is one
`set_registered_model_alias` call; building on an emulation that does not
document aliases would fork the promotion contract between the two parts.
Lifting the same Postgres-backed container into Azure Container Apps keeps
every row green in both environments, which is exactly what Part II does.

Off the critical path sit two exception tracks:
**[14 — Multi-GPU training](./14-multi-gpu-training.ipynb)**, the admitted escape
hatch when an LLM is *trained* rather than wrapped and outgrows the laptop, and
**[15 — Broker upgrade](./15-broker-upgrade.ipynb)**, used only if forced.
**[16 — End-to-end integration](./16-e2e-integration.ipynb)** then runs the identical
golden-path suite against both backends, closing the loop [chapter 08](08-environment-contract.ipynb)
opened.
